In [1]:
import torch.nn as nn
import torch
import torchvision.transforms.functional as func
import torch.optim as optim
from torch.amp import GradScaler, autocast

class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.conv2(x)
        x = self.relu(x)
        return x

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = UNetBlock(3,64)
        self.layer2 = UNetBlock(64,128)
        
        self.layer3 = UNetBlock(128, 256)
        self.up_3_to_4 = nn.ConvTranspose2d(256, 128, 2)
        
        self.layer4 = UNetBlock(256, 128)
        self.up_4_to_5 = nn.ConvTranspose2d(128, 64, 2)

        self.layer5 = UNetBlock(128, 64)
        self.output = nn.Conv2d(64,1,1)
        
        self.pool = nn.MaxPool2d(2)
    
    def forward(self, input):
        enc1 = self.layer1(input)
        enc2 = self.layer2(self.pool(enc1))
        
        bottleneck = self.layer3(self.pool(enc2))
        
        bottleneck_up = self.up_3_to_4(bottleneck)
        enc2_cropped = func.center_crop(enc2, [bottleneck_up.shape[2], bottleneck_up.shape[3]])
        dec1 = self.layer4(torch.cat([enc2_cropped, bottleneck_up], dim=1))
        
        dec1_up = self.up_4_to_5(dec1)
        enc1_cropped = func.center_crop(enc1, [dec1_up.shape[2], dec1_up.shape[3]])
        dec2 = self.layer5(torch.cat([enc1_cropped, dec1_up], dim=1))

        output = self.output(dec2)

        return output

In [2]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func
import torch.optim as optim
from torch.amp import GradScaler, autocast

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)

        self.initial = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool
        )
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4

    def forward(self, x):
        x0 = self.initial(x)
        x1 = self.encoder1(x0)
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x1)

        out = self.final_conv(d1)
        return out

In [3]:
from torch.utils.data import Dataset
import tifffile as tif
import os
from torchvision import transforms

imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
maskTransform = transforms.Compose([transforms.ToTensor()])

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (maskTransform(mask) > 0).float()

        return img, mask
    
def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

path = "../data/"
regions = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]

subdataset_to_region = {
    "Hokkaido Iburi-Tobu": "Hokkaido Iburi-Tobu",
    "Jiuzhai valley (UAV-0.2m)": "Jiuzhai Valley",
    "Jiuzhai valley (UAV-0.5m)": "Jiuzhai Valley",
    "Lombok": "Lombok",
    "Longxi River (SAT)": "Longxi River",
    "Longxi River (UAV)": "Longxi River",
    "Mengdong Township": "Mengdong Township",
    "Moxi town (UAV-0.2m)": "Luding",
    "Moxi town (UAV-1m)": "Luding",
    "Moxitaidi (SAT)": "Luding",
    "Moxitaidi (UAV-0.6m)": "Luding",
    "Moxitaidi (UAV-1m)": "Luding",
    "palu": "Palu",
    "Tiburon Peninsula (planet)": "Tiburon Peninsula",
    "Tiburon Peninsula (Sentinel)": "Tiburon Peninsula",
    "Wenchuan": "Wenchuan"
}

regions_dict = {
    "Hokkaido Iburi-Tobu": [],
    "Jiuzhai Valley": [],
    "Lombok": [],
    "Longxi River": [],
    "Mengdong Township": [],
    "Luding": [],
    "Palu": [],
    "Tiburon Peninsula": [],
    "Wenchuan": []
}

for region in regions:
    if(region != "study areas shp"):
        dataset_dir = "../data/" + region
        image_dir = os.path.join(dataset_dir, "img")
        img_list = os.listdir(image_dir)
        
        all_images = sorted(os.path.join(image_dir, f) for f in img_list)
        
        regions_dict[subdataset_to_region[region]].extend(all_images)

cross_regions_dict = {
    "Hokkaido Iburi-Tobu": [],
    "Jiuzhai Valley": [],
    "Lombok": [],
    "Longxi River": [],
    "Mengdong Township": [],
    "Luding": [],
    "Palu": [],
    "Tiburon Peninsula": [],
    "Wenchuan": []
}

for region in cross_regions_dict:
    for other_region in cross_regions_dict:
        if other_region != region:
            cross_regions_dict[region].extend(regions_dict[other_region])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))

pos_weight = torch.tensor([4.5]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [ ]:
### INDIVIDUAL REGION TRAINING AND TESTING

from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import random

random.seed(42)
batch_size = 8

output = "region,precision,recall,f1,iou,miou,oa"

for region in regions_dict:
    if len(regions_dict[region]) > 1000:
        regions_dict[region] = random.sample(regions_dict[region], 1000)
    
    image_paths = regions_dict[region]
    mask_paths = [f.replace("img", "mask") for f in image_paths]
    
    train_img, temp_img, train_mask, temp_mask = train_test_split(
        image_paths, mask_paths, test_size=.3, random_state=42
    )
    
    test_img, val_img, test_mask, val_mask = train_test_split(
        temp_img, temp_mask, test_size=.5, random_state=42
    )

    train_dataset = LandslideDataset(train_img, train_mask)
    val_dataset = LandslideDataset(val_img, val_mask)
    test_dataset = LandslideDataset(test_img, test_mask)
    
    trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)
    
    epochs = 40
    
    best_iou = 0.0
    patience = 10
    counter = 0
        
    model_path = "../models/baseline/individual/" + region + ".pth"
    
    model = ResNetUNet()
    # model = UNet()
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    scaler = GradScaler()
    
    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa')
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_num = 0

        for i, data in enumerate(trainLoader, 0):
            image, mask = data
            
            image = image.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()
            
            with autocast(device_type="cuda"):
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                bce = criterion(outputs, mask)
                dice = dice_loss(outputs, mask)
                loss = bce + dice
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            train_num += 1
        
        model.eval()
        val_loss = 0.0
        val_num = 0
        
        TP, FP, FN, TN = 0,0,0,0
        
        with torch.no_grad():
            for data in valLoader:
                image, mask = data

                image = image.to(device)
                mask = mask.to(device)
            
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                loss = criterion(outputs, mask)
                val_loss += loss.item()

                preds = torch.sigmoid(outputs) > .6
                
                preds_flat = preds.view(-1)
                mask_flat = mask.view(-1)
                
                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
        
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
        
        print(f'{region}, {epoch+1}, {running_loss / train_num :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break
    
    model.load_state_dict(torch.load(model_path))
    
    TP, FP, FN, TN = 0,0,0,0
    
    for data in testLoader:
        image, mask = data

        image = image.to(device)
        mask = mask.to(device)
            
        outputs = model(image)
        outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)

        preds = torch.sigmoid(outputs) > .6
                
        preds_flat = preds.view(-1)
        mask_flat = mask.view(-1)
                
        TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
        FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
        FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
        TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
        
    precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
    output += f'\n{region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'

with open("../results/baseline/individual.csv", "w") as f:
    f.write(output)

In [ ]:
### PAIRWISE TRANSFER

from torch.utils.data import DataLoader
import random

random.seed(42)
batch_size = 8

model = ResNetUNet()
# model = UNet()
model.to(device)
model.eval()

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

for source_region in regions_dict:
    model_path = "../models/baseline/individual/" + source_region + ".pth"
    model.load_state_dict(torch.load(model_path))
    
    print(f'source_region, target_region, precision, recall, f1, iou, miou, oa')
    
    for target_region in regions_dict:
        if source_region != target_region:
        
            TP, FP, FN, TN = 0,0,0,0
            
            if len(regions_dict[target_region]) > 1000:
                regions_dict[target_region] = random.sample(regions_dict[target_region], 1000)
            
            image_paths = regions_dict[target_region]
            mask_paths = [f.replace("img", "mask") for f in image_paths]
            
            test_dataset = LandslideDataset(image_paths, mask_paths)
            testLoader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
            for data in testLoader:
                image, mask = data

                image = image.to(device)
                mask = mask.to(device)
                    
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)

                preds = torch.sigmoid(outputs) > .6
                        
                preds_flat = preds.view(-1)
                mask_flat = mask.view(-1)
                        
                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
        
            precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
            output += f'\n{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')


with open("../results/baseline/pairwise_transfer.csv", "w") as f:
    f.write(output)

In [4]:
### LOO-CV
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import random

random.seed(42)
batch_size = 32

output = "region,precision,recall,f1,iou,miou,oa"

for region in cross_regions_dict:    
    image_paths = cross_regions_dict[region]
    mask_paths = [f.replace("img", "mask") for f in image_paths]
    
    train_img, temp_img, train_mask, temp_mask = train_test_split(
        image_paths, mask_paths, test_size=.3, random_state=42
    )
    
    test_img, val_img, test_mask, val_mask = train_test_split(
        temp_img, temp_mask, test_size=.5, random_state=42
    )

    train_dataset = LandslideDataset(train_img, train_mask)
    val_dataset = LandslideDataset(val_img, val_mask)
    test_dataset = LandslideDataset(test_img, test_mask)
    
    trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)
    
    epochs = 40
    
    best_iou = 0.0
    patience = 10
    counter = 0
        
    model_path = "../models/baseline/loocv/" + region + ".pth"
    
    model = ResNetUNet()
    # model = UNet()
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

    scaler = GradScaler()
    
    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa')
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_num = 0

        for i, data in enumerate(trainLoader, 0):
            image, mask = data
            
            image = image.to(device)
            mask = mask.to(device)

            optimizer.zero_grad()
            
            with autocast(device_type="cuda"):
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                bce = criterion(outputs, mask)
                dice = dice_loss(outputs, mask)
                loss = bce + dice
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            train_num += 1
        
        model.eval()
        val_loss = 0.0
        val_num = 0
        
        TP, FP, FN, TN = 0,0,0,0
        
        with torch.no_grad():
            for data in valLoader:
                image, mask = data

                image = image.to(device)
                mask = mask.to(device)
            
                outputs = model(image)
                outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)
                loss = criterion(outputs, mask)
                val_loss += loss.item()

                preds = torch.sigmoid(outputs) > .6
                
                preds_flat = preds.view(-1)
                mask_flat = mask.view(-1)
                
                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
        
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
        
        print(f'{region}, {epoch+1}, {running_loss / train_num :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break
    
    model.load_state_dict(torch.load(model_path))
    
    TP, FP, FN, TN = 0,0,0,0
    
    for data in testLoader:
        image, mask = data

        image = image.to(device)
        mask = mask.to(device)
            
        outputs = model(image)
        outputs = nn.functional.interpolate(outputs, size=mask.shape[2:], mode="bilinear", align_corners=False)

        preds = torch.sigmoid(outputs) > .6
                
        preds_flat = preds.view(-1)
        mask_flat = mask.view(-1)
                
        TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
        FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
        FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
        TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
        
    precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)
    output += f'\n{region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'

with open("../results/baseline/loocv.csv", "w") as f:
    f.write(output)

region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, 1, 1.098, 0.544, 0.6252, 0.7198, 0.6692, 0.5028, 0.6947, 0.8982
Hokkaido Iburi-Tobu, 2, 0.907, 0.497, 0.6677, 0.7538, 0.7081, 0.5481, 0.7243, 0.9112
Hokkaido Iburi-Tobu, 3, 0.851, 0.475, 0.5149, 0.9112, 0.6580, 0.4903, 0.6673, 0.8646
Hokkaido Iburi-Tobu, 4, 0.816, 0.448, 0.6728, 0.7662, 0.7165, 0.5582, 0.7304, 0.9133
Hokkaido Iburi-Tobu, 5, 0.780, 0.388, 0.6571, 0.8415, 0.7379, 0.5847, 0.7438, 0.9145
Hokkaido Iburi-Tobu, 6, 0.746, 0.431, 0.7501, 0.7592, 0.7546, 0.6060, 0.7634, 0.9294
Hokkaido Iburi-Tobu, 7, 0.724, 0.416, 0.7578, 0.7569, 0.7573, 0.6094, 0.7658, 0.9307
Hokkaido Iburi-Tobu, 8, 0.690, 0.387, 0.6797, 0.8561, 0.7577, 0.6100, 0.7604, 0.9217
Hokkaido Iburi-Tobu, 9, 0.648, 0.341, 0.6732, 0.8862, 0.7652, 0.6197, 0.7653, 0.9222
Hokkaido Iburi-Tobu, 10, 0.598, 0.339, 0.6796, 0.8919, 0.7714, 0.6279, 0.7706, 0.9244
Hokkaido Iburi-Tobu, 11, 0.517, 0.296, 0.7055, 0.9032, 0.7922, 0.6559, 0.78